[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nattaponm/Teach_stormTracks_KKN_May2024_HID_doppler/blob/main/prep/P3_tracking_month.ipynb)


# P3 — ติดตามพายุทั้งเดือน (T-DaTing 15 / 10 / 5 นาที) ตารางเซลล์ เส้นทาง footprint และกริดสรุป

**สำหรับผู้ดูแลข้อมูล** — ต้องรัน P1 และ P2 ก่อน (ไม่อ่าน UF แล้ว ใช้เฉพาะไฟล์ derived)

ใช้ฟังก์ชันของ repo3 (`track_tools`) และพารามิเตอร์ T-DaTing เดิมทุกค่า (`minref=33, maxref=45, mindiff=5, minsize=15, minmax=40, mindis=8, mintrack=2, match_frac=0.25`) ส่วนที่เพิ่มอยู่ใน `utils/tracking.py`:
- ติดตามเป็น **ช่วงต่อเนื่อง (segment)** ช่องเวลาที่ขาดตัดเส้นทาง และสร้างภาพ 5/10 นาที (advection interpolation แบบ linZ) เฉพาะภายใน segment
- ตัดเฉพาะช่วงที่มีพายุ (พื้นที่ ≥ 35 dBZ ≥ 20 km²) ± 30 นาที
- เส้นทางที่เริ่มหรือจบที่ขอบ segment ติดป้าย `censored_start` / `censored_end`
- `uid` = YYYYMMDD-ttt ไม่ซ้ำทั้งเดือน

| ผลลัพธ์ | ที่เก็บ |
|---|---|
| `tracks/KKN_YYYYMMDD_labels_<tag>.nc` (label cube รายวัน) | Drive → Zenodo |
| `cells_<tag>.csv.gz`, `tracks_<tag>.csv`, `footprints_<tag>.gpkg`, `segments.csv`, `tracking_summary.csv` | `data/derived/` (ทั้งเดือน) |
| `terrain_1km.nc` (ภูมิประเทศแบบ repo3 จาก Copernicus DEM), `monthly_summary_1km.nc` (ความถี่ echo, ความหนาแน่นเส้นทาง, ความถี่ลูกเห็บ) | `data/derived/` |

In [ ]:
# @title ⚙️ เตรียมสภาพแวดล้อม (รันก่อนเสมอ)
import os, sys, subprocess
REPO = "Teach_stormTracks_KKN_May2024_HID_doppler"
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    if not os.path.isdir(f"/content/{REPO}"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", f"https://github.com/nattaponm/{REPO}.git", f"/content/{REPO}"], check=True)
    else:
        subprocess.run(["git", "-C", f"/content/{REPO}", "pull", "-q"])
    os.chdir(f"/content/{REPO}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pysteps==1.21.5", "opencv-python-headless"], check=True)
elif os.path.basename(os.getcwd()) in ("notebooks", "prep"):
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("โฟลเดอร์ทำงาน:", os.getcwd(), "| CPU", os.cpu_count(), "core")

In [ ]:
import json, time, shutil, warnings
from pathlib import Path
from functools import partial
from concurrent.futures import ProcessPoolExecutor
import multiprocessing as mproc
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
warnings.filterwarnings("ignore")

from utils.config import load_settings, repo_path
from utils import maps as M
s = load_settings()
M.setup_style()
OUT_ROOT = Path(os.environ.get("KKN_OUT_ROOT", s["radar_data"]["drive_derived_root"]))
N_WORKERS = os.cpu_count()
CTX = mproc.get_context("fork")

from utils import tracking as TR, track_tools as tt, data_io, derived_io as DIO, geo as G
CAPPI_DIR, POLAR_DIR, TRACK_DIR = OUT_ROOT / "cappi", OUT_ROOT / "polar", OUT_ROOT / "tracks"
TRACK_DIR.mkdir(parents=True, exist_ok=True)
DER = repo_path("data", "derived"); DER.mkdir(parents=True, exist_ok=True)
TAGS = {k: TR.MAIN_TAGS[k] for k in s["tracking"]["tags"]}
DAYS = None             # None = ทุกวันที่มีไฟล์ CAPPI | หรือระบุ เช่น ["20240503"]
OVERWRITE = False
cappi_files = sorted(CAPPI_DIR.glob("KKN_*_cappi_15min.nc"))
days = [p.name[4:12] for p in cappi_files]
days = days if DAYS is None else [d for d in DAYS if d in days]
print(f"{len(days)} วัน:", days)

## 1. กริดภูมิประเทศ (Copernicus GLO-90 → 1 km) และความสูงลำคลื่น

สูตรเดียวกับ repo3: elevation, slope, relief (ช่วงสูงต่ำในหน้าต่าง 11 km), $\partial h/\partial x$, $\partial h/\partial y$, TPI (ความสูงเทียบค่าเฉลี่ย 11 km)

In [ ]:
_, meta = data_io.load_pcappi(str(cappi_files[0]))
dem_path = repo_path(s["dem"]["out"])
if not dem_path.exists():
    G.build_dem(s, out=dem_path)
TERR = TR.terrain_from_dem(dem_path, meta)
BEAM = tt.beam_height_grid(meta)
DIO.write_grids_2d(DER / "terrain_1km.nc", {**TERR, "beam_h0_km": BEAM}, meta,
                   {"source": "Copernicus GLO-90 averaged to 1 km (NB01); formulas as repo3 make_terrain_grid",
                    "beam_elevation_deg": tt.ELEV0})
print({k: f"{np.nanmin(v):.1f}–{np.nanmax(v):.1f}" for k, v in TERR.items()})

## 2. ติดตามทีละวัน (ขนานระหว่างวัน)

In [ ]:
def run_day(day):
    t0 = time.time()
    Z, mt = data_io.load_pcappi(str(CAPPI_DIR / f"KKN_{day}_cappi_15min.nc"))
    av = mt["available"]
    obs = {"n_obs": av.sum(), "freq35": np.nansum(Z[av] >= 35, axis=0), "freq45": np.nansum(Z[av] >= 45, axis=0)}
    pf = POLAR_DIR / f"KKN_{day}_polar_15min.nc"
    P, PT = tt.load_polar_grid(str(pf)) if pf.exists() else (None, None)
    hail = P["hail_col"].sum(0).astype(np.int32) if P is not None else np.zeros_like(obs["freq35"])
    res, seg = TR.track_day(day, Z, mt, P, PT, TERR, BEAM, tags=TAGS, thr_dbz=s["tracking"]["thr_dbz"],
                            min_area_km2=s["tracking"]["min_area_km2"])
    out = {"day": day, "seg": seg, "obs": obs, "hail": hail, "tags": {}}
    for tag, r in res.items():
        if r is None:
            continue
        DIO.write_labels(TRACK_DIR / f"KKN_{day}_labels_{tag}.nc", r["labels"], r["times"], r["observed"], mt,
                         {"params": str(tt.PARAMS), "track_kw": str(tt.TRACK_KW), "tag": tag})
        keep = set(r["tracks"].index[r["tracks"]["lifetime_min"] >= s["tracking"]["min_life_min"]])
        origin = np.zeros(mt["coverage"].shape, np.int32)
        for k, g in r["cells"].groupby("track_id"):
            if k in keep:
                g = g.sort_values("time")
                ci = int(np.clip(round((g["cx"].iloc[0] - mt["x"][0]) / 1000), 0, origin.shape[1] - 1))
                ri = int(np.clip(round((mt["y"][0] - g["cy"].iloc[0]) / 1000), 0, origin.shape[0] - 1))
                origin[ri, ci] += 1
        out["tags"][tag] = {"cells": r["cells"], "tracks": r["tracks"].reset_index(),
                            "footprints": r["footprints"], "density": TR.track_density(r["fp_masks"], origin.shape, keep),
                            "origin": origin}
    out["sec"] = time.time() - t0
    return out

t_all = time.time()
results = []
with ProcessPoolExecutor(max(1, N_WORKERS // 2), mp_context=CTX) as ex:     # T-DaTing ใช้หน่วยความจำมาก → ครึ่งหนึ่งของ core
    for r in ex.map(run_day, days):
        n = {t: len(v["tracks"]) for t, v in r["tags"].items()}
        print(f"{r['day']}: {len(r['seg'])} segment | เส้นทาง {n} | {r['sec'] / 60:.1f} นาที")
        results.append(r)
print(f"รวม {(time.time() - t_all) / 60:.1f} นาที")

## 3. รวมทั้งเดือน → ตาราง, footprint, กริดสรุป

In [ ]:
import geopandas as gpd
pd.concat([r["seg"] for r in results], ignore_index=True).to_csv(DER / "segments.csv", index=False)
summary = []
for tag in TAGS:
    parts = [r["tags"][tag] for r in results if tag in r["tags"]]
    if not parts:
        continue
    C = pd.concat([p["cells"] for p in parts], ignore_index=True)
    T = pd.concat([p["tracks"] for p in parts], ignore_index=True)
    F = pd.concat([p["footprints"] for p in parts], ignore_index=True)
    T["phase"] = pd.cut(T["day"].dt.day, [0, 9, 20, 31], labels=[p["key"] for p in s["phases"]])
    C.to_csv(DER / f"cells_{tag}.csv.gz", index=False, float_format="%.5g")
    T.to_csv(DER / f"tracks_{tag}.csv", index=False, float_format="%.5g")
    F = gpd.GeoDataFrame(F, geometry="geometry", crs=F.crs)
    F["day"] = F["day"].astype(str)
    F.to_file(DER / f"footprints_{tag}.gpkg", driver="GPKG")
    life = T["lifetime_min"]
    summary.append({"tag": tag, "days": T["day"].nunique(), "tracks": len(T), "life>=30min": int((life >= 30).sum()),
                    "life>=60min": int((life >= 60).sum()), "censored": int((T["censored_start"] | T["censored_end"]).sum()),
                    "max_life_min": life.max()})
summary = pd.DataFrame(summary); summary.to_csv(DER / "tracking_summary.csv", index=False)
summary

In [ ]:
n_obs = sum(int(r["obs"]["n_obs"]) for r in results)
grids = {"n_obs_total": np.full(meta["coverage"].shape, n_obs, np.float32),
         "freq_ge35_pct": 100 * sum(r["obs"]["freq35"] for r in results) / max(n_obs, 1),
         "freq_ge45_pct": 100 * sum(r["obs"]["freq45"] for r in results) / max(n_obs, 1),
         "hail_col_count": sum(r["hail"] for r in results).astype(np.float32)}
for tag in TAGS:
    grids[f"track_density_{tag}"] = sum(r["tags"][tag]["density"] for r in results if tag in r["tags"]).astype(np.float32)
    grids[f"origin_count_{tag}"] = sum(r["tags"][tag]["origin"] for r in results if tag in r["tags"]).astype(np.float32)
for k in grids:
    grids[k] = np.where(meta["coverage"], grids[k], np.nan)
DIO.write_grids_2d(DER / "monthly_summary_1km.nc", grids, meta,
                   {"days": ",".join(days), "n_observed_volumes": n_obs,
                    "note": "freq = % of observed 15-min volumes; track density/origins count tracks with lifetime >= "
                            f"{s['tracking']['min_life_min']} min; hail_col_count counts storm-active volumes only (P2)"})
print("บันทึก monthly_summary_1km.nc:", list(grids))

## 4. ตรวจเทียบกับ repo3 (3 พ.ค., เส้นทางที่เริ่มระหว่าง 05–17 UTC)

repo3 ติดตามเฉพาะ 05–17 UTC ผลจึงต่างกันได้เล็กน้อยที่ขอบเวลา (เส้นทางที่ต่อเนื่องจากก่อน 05 UTC หรือหลัง 17 UTC) แต่จำนวนควรใกล้เคียง: repo3 = 114 เส้นทาง, ≥ 30 นาที 59 เส้นทาง (15 นาที)

In [ ]:
p = DER / "tracks_15min.csv"
if p.exists():
    T15 = pd.read_csv(p, parse_dates=["day", "start", "end"])
    d3 = T15[(T15["day"] == "2024-05-03") & (T15["start"].dt.hour >= 5) & (T15["start"] <= "2024-05-03 17:00")]
    print(f"3 พ.ค. เริ่ม 05–17 UTC: {len(d3)} เส้นทาง, ≥ 30 นาที {(d3['lifetime_min'] >= 30).sum()} (repo3: 114 / 59)")

## 5. แผนที่เส้นทางทั้งเดือน (≥ 30 นาที, 15 นาที) ระบายสีตามช่วงของเดือน

In [ ]:
C15 = pd.read_csv(DER / "cells_15min.csv.gz", parse_dates=["day", "time"])
T15 = pd.read_csv(DER / "tracks_15min.csv", parse_dates=["day"])
long_ = set(T15.loc[T15["lifetime_min"] >= s["tracking"]["min_life_min"], "uid"])
fig, ax = plt.subplots(figsize=(7.2, 7.2))
dem_z, dem_ext = G.load_dem(s, build_if_missing=False)
M.base_map(ax, s, dem=(dem_z, dem_ext), dem_alpha=0.6)
for uid, g in C15[C15["uid"].isin(long_)].groupby("uid"):
    g = g.sort_values("time")
    ph = "outbreak" if g["day"].iloc[0].day <= 9 else "late_pre" if g["day"].iloc[0].day <= 20 else "rainy"
    ax.plot(g["cx"] / 1e3, g["cy"] / 1e3, color=M.PHASE_COLORS[ph], lw=0.7, alpha=0.8, zorder=12)
    ax.plot(g["cx"].iloc[0] / 1e3, g["cy"].iloc[0] / 1e3, "o", ms=2, mfc="white", mec=M.PHASE_COLORS[ph], mew=0.6, zorder=13)
from matplotlib.lines import Line2D
ax.legend([Line2D([], [], color=c) for c in M.PHASE_COLORS.values()], [p["label_th"] for p in s["phases"]],
          loc="lower right", title=f"เส้นทาง ≥ {s['tracking']['min_life_min']} นาที (○ = จุดเริ่ม)", title_fontsize=7)
ax.set_title(f"เส้นทางพายุ T-DaTing 15 นาที ทั้งเดือน (n = {len(long_)})", loc="left")
M.savefig(fig, "p3_tracks_month", outdir=OUT_ROOT / "figures")
plt.show()

**ขั้นต่อไป:** รัน P4 เพื่อจัดแพ็ก ตรวจ checksum คัดลอกวันหลักเข้า repo และเตรียมอัปโหลด Zenodo